# MedGemma 4B API Server on Kaggle

This notebook sets up a **FastAPI server** wrapping the **MedGemma 4B** model, exposed via **ngrok** for remote access.

## Architecture

```
Your Local CLI → HTTPS (ngrok) → Kaggle Notebook (FastAPI + MedGemma 4B on GPU)
```

## Prerequisites
- Kaggle GPU runtime (T4 or better)
- Internet enabled (for model download & ngrok)
- Hugging Face token with access to the gated model, stored in Kaggle Secrets as `HF_TOKEN`
- ngrok authtoken (stored in Kaggle Secrets as `NGROK_AUTHTOKEN`)
> **Version:** `2026-11-04 00:14`  |  **Commit:** `b73e960`  |  **Fix:** pinned transformers==4.45.2 / huggingface_hub==0.25.2


---
## 🔧 SECTION 1: INITIALIZATION & ENVIRONMENT SETUP

Run this section **once per session** to configure the environment.

In [ ]:
# 1.1 Install system dependencies
# Pin compatible versions: transformers>=4.46 for AutoModelForImageTextToText,
# huggingface_hub>=0.26 for _httpx2_follow_hub_redirects_with_backoff
%pip install -q "transformers==4.48.0" "accelerate==0.34.2" "huggingface_hub==0.26.2"
%pip install -q fastapi uvicorn pydantic nest-asyncio pyngrok
%pip install -q pillow requests

---
## 🔐 SECTION 2: AUTHENTICATION

Run once to authenticate with Hugging Face and configure ngrok.

In [ ]:
# 2.1 Hugging Face Login (for gated MedGemma model)
# Prefers HF_TOKEN from Kaggle → Settings → Secrets.
# The token must belong to an account that accepted the license at
# https://huggingface.co/google/medgemma-4b-it
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
try:
    login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
    print("✅ Hugging Face authenticated via HF_TOKEN secret")
except Exception as e:
    print(f"⚠️ HF_TOKEN secret unavailable ({e}); falling back to interactive login")
    login()  # Opens browser for device code flow

# 2.2 ngrok Authentication (from Kaggle Secrets)
# Add NGROK_AUTHTOKEN in Kaggle → Settings → Secrets
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)
print("✅ ngrok authenticated")

---
## 🤖 SECTION 3: LOAD MEDGEMMA MODEL

Load the model and processor. **Run this once per session** - takes ~1-2 minutes.

In [ ]:
# 3.1 Configuration
MODEL_ID = "google/medgemma-4b-it"

# 3.2 Load processor
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained(MODEL_ID)

# 3.3 Load model with optimal dtype
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    attn_implementation="flash_attention_2" if torch.cuda.is_bf16_supported() else "eager",
)

model.eval()

print(f"✅ Model loaded: {MODEL_ID}")
print(f"Precision: {dtype}")
print(f"Device: {model.device}")
print(f"GPU memory allocated: {round(torch.cuda.memory_allocated() / 1024**3, 2)} GiB")

---
## 🧪 SECTION 4: QUICK LOCAL TEST (Optional)

Verify the model works before starting the API server.

In [ ]:
# Quick inference test
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "What is iron deficiency anemia in one sentence?"}
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
answer = processor.decode(new_tokens, skip_special_tokens=True)

print("--- TEST RESPONSE ---")
print(answer)

---
## 🌐 SECTION 5: FASTAPI SERVER SETUP

Define the API with authentication, then start it in a background thread.

In [ ]:
# 5.1 Apply nest_asyncio FIRST (required for Jupyter/Kaggle)
import nest_asyncio
nest_asyncio.apply()
print("✅ nest_asyncio applied")

# 5.2 Imports
import os
import secrets
import threading
import time
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
import uvicorn

# 5.3 Generate/load API key
if "MEDGEMMA_API_KEY" not in os.environ:
    os.environ["MEDGEMMA_API_KEY"] = secrets.token_urlsafe(32)

API_KEY = os.environ["MEDGEMMA_API_KEY"]

# 5.4 FastAPI app definition
app = FastAPI(
    title="Private MedGemma API",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc",
)

class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=12000)
    max_new_tokens: int = Field(default=512, ge=1, le=2048)
    temperature: float = Field(default=0.0, ge=0.0, le=2.0)
    top_p: float = Field(default=1.0, ge=0.0, le=1.0)

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": "google/medgemma-4b-it",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "precision": str(dtype),
    }

@app.post("/generate")
def generate(
    request: GenerateRequest,
    x_api_key: str = Header(default=""),
):
    # Constant-time comparison for security
    if not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Invalid API key")

    # Format as chat
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": request.prompt},
            ],
        }
    ]

    # Tokenize
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    # Generate
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=request.max_new_tokens,
            do_sample=request.temperature > 0,
            temperature=request.temperature if request.temperature > 0 else None,
            top_p=request.top_p if request.temperature > 0 else None,
        )

    # Decode only new tokens
    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    answer = processor.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    return {
        "model": "google/medgemma-4b-it",
        "response": answer,
    }

print("✅ FastAPI app defined")

In [ ]:
# 5.5 Start API server in background thread
def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",  # Must bind to all interfaces for ngrok
        port=8000,
        log_level="warning",
    )

# Kill any existing process on port 8000
import subprocess
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(0.5)

# Start server thread
if "api_thread" not in globals() or not api_thread.is_alive():
    api_thread = threading.Thread(target=run_api, daemon=True)
    api_thread.start()
    time.sleep(2)  # Wait for server to start
    print("✅ API server started in background")
else:
    print("⚠️ API server already running")

print(f"🔑 API Key: {API_KEY}")
print(f"📍 Local API: http://0.0.0.0:8000")
print(f"📚 Swagger UI: http://0.0.0.0:8000/docs")

---
## 🔗 SECTION 6: NGROK TUNNEL

Expose the local API to the internet. **Run after API server is confirmed running.**

In [ ]:
# 6.1 Create ngrok tunnel to port 8000
from pyngrok import ngrok

# Close any existing tunnels to port 8000
for tunnel in ngrok.get_tunnels():
    if "8000" in str(tunnel.config.get("addr", "")):
        ngrok.disconnect(tunnel.public_url)
        print(f"Closed existing tunnel: {tunnel.public_url}")

# Open new tunnel
tunnel = ngrok.connect(8000, "http")

NGROK_URL = tunnel.public_url
print(f"🌐 Public URL: {NGROK_URL}")
print(f"📋 Health check: {NGROK_URL}/health")
print(f"📋 Generate endpoint: {NGROK_URL}/generate")
print(f"📚 Swagger UI: {NGROK_URL}/docs")

# Save for later cells
os.environ["MEDGEMMA_API_URL"] = NGROK_URL

---
## ✅ SECTION 7: VERIFICATION & TESTING

Test the complete pipeline: local → ngrok → FastAPI → MedGemma

In [ ]:
# 7.1 Test health endpoint
import requests

url = os.environ.get("MEDGEMMA_API_URL")
headers = {"X-API-Key": API_KEY}

print("Testing health endpoint...")
response = requests.get(f"{url}/health", timeout=30)
print(f"Status: {response.status_code}")
print(response.json())

In [ ]:
# 7.2 Test generation endpoint
print("Testing generation endpoint...")
response = requests.post(
    f"{url}/generate",
    headers=headers,
    json={
        "prompt": "Explain the difference between iron and vitamin B12 deficiency in three bullet points.",
        "max_new_tokens": 300,
        "temperature": 0.0,
    },
    timeout=180,
)

print(f"Status: {response.status_code}")
result = response.json()
print(f"Model: {result.get('model')}")
print(f"\nResponse:\n{result.get('response')}")

---
## 💻 SECTION 8: LOCAL CLIENT USAGE

Copy your API key and ngrok URL to your local machine and use the CLI:

```bash
# On your local machine
export MEDGEMMA_API_URL="https://YOUR-NGROK-URL.ngrok-free.app"
export MEDGEMMA_API_KEY="YOUR-API-KEY-FROM-ABOVE"

# Health check
python3 medgemma.py health

# Single question
python3 medgemma.py ask "What are the symptoms of iron deficiency?"

# Interactive chat
python3 medgemma.py chat
```

---
## 🔄 SECTION 9: RESTART / RECOVERY

If the kernel restarts or session expires, run these cells in order:

In [ ]:
# RESTART RECOVERY - Run these in order after kernel restart

# 1. Re-apply nest_asyncio (MUST BE FIRST)
import nest_asyncio
nest_asyncio.apply()

# 2. Re-authenticate with Hugging Face (credentials are lost on restart)
#    Requires HF_TOKEN in Kaggle → Settings → Secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
print("✅ Hugging Face re-authenticated")

# 3. Re-load model (if not in memory)
if "model" not in globals() or "processor" not in globals():
    print("Reloading model...")
    from transformers import AutoProcessor, AutoModelForImageTextToText
    import torch
    
    MODEL_ID = "google/medgemma-4b-it"
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map="auto",
    )
    model.eval()
    print("✅ Model reloaded")
else:
    print("✅ Model already in memory")

In [ ]:
# 4. Restart API server (if needed)
import threading
import time
import subprocess
import os
import uvicorn
import secrets

# Kill old process
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(0.5)

# Reuse existing API_KEY from environment or regenerate
API_KEY = os.environ.get("MEDGEMMA_API_KEY", secrets.token_urlsafe(32))
os.environ["MEDGEMMA_API_KEY"] = API_KEY

# Restart server thread
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()
time.sleep(2)
print(f"✅ API restarted. Key: {API_KEY}")

In [ ]:
# 5. Re-establish ngrok tunnel
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import os, requests, time

# Re-authenticate
user_secrets = UserSecretsClient()
ngrok_token = user_secrets.get_secret("NGROK_AUTHTOKEN")
try:
    ngrok.set_auth_token(ngrok_token)
    print("✅ ngrok authenticated")
except Exception as e:
    print(f"⚠️ ngrok set_auth_token warning (continuing): {e}")

# Clean old tunnels
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

# Create new tunnel
tunnel = ngrok.connect(8000, "http")
NGROK_URL = tunnel.public_url
os.environ["MEDGEMMA_API_URL"] = NGROK_URL

print(f"🌐 New public URL: {NGROK_URL}")
print(f"🔑 API Key: {API_KEY}")

# Quick verify - safe JSON handling, with tunnel retry
health_ok = False
for attempt in range(3):
    r = requests.get(f"{NGROK_URL}/health", headers={"X-API-Key": API_KEY}, timeout=60)
    print("Health check status:", r.status_code)
    print("Health check body:", r.text)
    if r.status_code == 200:
        health_ok = True
        print("Health check OK:", r.json())
        break
    print(f"retrying: attempt {attempt + 1}/3, recreating tunnel...")
    time.sleep(10)
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    tunnel = ngrok.connect(8000, "http")
    NGROK_URL = tunnel.public_url
    os.environ["MEDGEMMA_API_URL"] = NGROK_URL

if not health_ok:
    raise RuntimeError("ngrok tunnel failed to become healthy after retries")


---
## 📋 SUMMARY: EXECUTION ORDER

| Step | Section | When to Run |
|------|---------|-------------|
| 1 | **Initialization** | Every session start |
| 2 | **Authentication** | Once, then comment out |
| 3 | **Load Model** | Every session start (~2 min) |
| 4 | **Local Test** | Optional, after model load |
| 5 | **FastAPI Setup** | Every session start |
| 6 | **ngrok Tunnel** | After API server runs |
| 7 | **Verification** | After tunnel created |
| 8 | **Local CLI** | On your local machine |
| 9 | **Restart/Recovery** | After kernel restart/session expiry |

### ⚠️ Important Notes
- **Hugging Face on restart**: session credentials do not survive a kernel restart. Section 9 re-authenticates from the `HF_TOKEN` secret before reloading the model. The token must belong to an account that accepted the license at https://huggingface.co/google/medgemma-4b-it, otherwise you'll still get a `401 GatedRepoError`.
- **ngrok free tier**: URL changes every restart. Update `MEDGEMMA_API_URL` locally.
- **Kaggle timeout**: Session stops after ~60 min inactivity. Use "Keep session alive" in settings.
- **API key**: Regenerated on each full restart. Save it if you need consistency.
- **GPU memory**: MedGemma 4B uses ~5.5 GB. T4 (16 GB) handles it comfortably.